# Full-Stack Web Development: From Fundamentals to Advanced

This notebook teaches the concepts behind your full-stack rebuild of DRAM Express: HTTP, REST API design, FastAPI, React, and the client-server architecture connecting them.

**How this notebook works:** this is a Python kernel, so the HTTP/backend sections run real, live requests against your actual FastAPI server. The React sections cannot execute JS here, so those are explained using the real code from your project, annotated, rather than run inline. Make sure your backend is running in a terminal first: `python -m uvicorn api:app --reload --port 8000`

**Structure:**
1. HTTP Fundamentals
2. REST API Design
3. FastAPI: Validation, Docs, and What Makes It Different
4. React Fundamentals
5. The Client-Server Boundary (the thing that actually broke, and why)
6. Sessions, State, and Security

## Setup

In [1]:
import requests
import json

API_BASE = "http://localhost:8000"

try:
    response = requests.get(f"{API_BASE}/docs")
    print(f"Backend is up. Status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("Backend is NOT reachable. Start it with: python -m uvicorn api:app --reload --port 8000")

Backend is up. Status: 200


---
# Part 1: HTTP Fundamentals

Every request between your React frontend and FastAPI backend is an HTTP request. Understanding what's inside one demystifies almost everything that follows.

## 1.1 What's Actually in an HTTP Request

An HTTP request has four parts:
- **Method** -- what kind of action (GET, POST, PUT, DELETE, ...)
- **URL/path** -- which resource
- **Headers** -- metadata (content type, auth tokens, etc.)
- **Body** -- the actual data being sent, usually JSON for POST/PUT

Let's make a real request to your API and inspect both the request and response.

In [2]:
payload = {"customer_id": 4, "password": "123"}

response = requests.post(
    f"{API_BASE}/api/login",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload),
)

print("--- REQUEST ---")
print("Method: POST")
print("URL:", f"{API_BASE}/api/login")
print("Body sent:", payload)

print("\n--- RESPONSE ---")
print("Status code:", response.status_code)
print("Response body:", response.json())

--- REQUEST ---
Method: POST
URL: http://localhost:8000/api/login
Body sent: {'customer_id': 4, 'password': '123'}

--- RESPONSE ---
Status code: 200
Response body: {'session_id': 'bca01d52-217b-4cab-8250-7a4d44b7bc94', 'first_name': 'Jesse'}


## 1.2 HTTP Methods and What They're Supposed to Mean

REST convention (not enforced by the protocol itself, but widely followed):

| Method | Meaning | Your project's usage |
|---|---|---|
| GET | Retrieve data, no side effects | `/api/orders` |
| POST | Create something, or perform an action with side effects | `/api/login`, `/api/chat`, `/api/review` |
| PUT/PATCH | Update an existing resource | Not used in this project |
| DELETE | Remove a resource | Not used in this project |

Notice `/api/chat` is a POST even though "asking a question" might feel read-only. It's POST because it has side effects (querying the database, generating a response) and sends a body, which GET requests conventionally don't carry.

## 1.3 Status Codes: What They Actually Communicate

- **2xx** -- success
- **4xx** -- the client made a mistake (bad input, not authorized, not found)
- **5xx** -- the server made a mistake

Let's trigger a couple of these on purpose.

In [3]:
bad_login = requests.post(f"{API_BASE}/api/login", json={"customer_id": 4, "password": "wrong"})
print("Bad password ->", bad_login.status_code, bad_login.json())

malformed = requests.post(f"{API_BASE}/api/login", json={"customer_id": "not a number", "password": "123"})
print("\nMalformed request ->", malformed.status_code)
print(malformed.json())

# Notice: you never wrote code to check "is customer_id actually a number?" --
# FastAPI's Pydantic validation did that automatically from the type hints
# in your LoginRequest class, before your endpoint function ever ran.

Bad password -> 401 {'detail': 'Login failed.'}

Malformed request -> 422
{'detail': [{'type': 'int_parsing', 'loc': ['body', 'customer_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not a number'}]}


---
# Part 2: REST API Design

REST isn't a protocol or a library, it's a set of conventions for organizing an API around **resources** (nouns) manipulated by HTTP methods (verbs).

## 2.1 Resources, Not Actions, in the URL

A REST URL generally describes *what thing* you're working with, not *what action* to take (the method already says that).

- Less RESTful: `/getOrders`, `/doLogin`, `/submitReview`
- More RESTful: `GET /orders`, `POST /login`, `POST /reviews`

Your project mostly follows this, though `/api/chat` is intentionally more action-flavored since "ask a question" doesn't map cleanly onto a resource the way orders/reviews do -- a common, reasonable exception in real APIs.

## 2.2 Why Login/Review Changed Shape for the API

Your terminal and Streamlit versions handled login and reviews as a **multi-turn conversation**: ask for customer ID, wait, ask for password, wait. Over REST, each request is independent and stateless by default, there's no built-in "wait for the next message in this exchange."

This is why the API version collects everything in one request instead.

In [4]:
login_response = requests.post(f"{API_BASE}/api/login", json={"customer_id": 4, "password": "123"})
session_id = login_response.json()["session_id"]
print("Got a session_id from ONE request:", session_id)

# Compare to the terminal version's flow:
#   Customer ID: [wait for input]
#   Password: [wait for input]
# That back-and-forth doesn't map naturally onto discrete HTTP requests -- which is
# exactly why LoginForm.jsx collects both fields before submitting once, rather than
# sending a separate request per field.

Got a session_id from ONE request: 97348c97-71dd-4d16-9579-5ff46440a75a


## 2.3 Request Validation with Pydantic

FastAPI uses Pydantic models to define exactly what shape of data an endpoint expects, and enforces it automatically before your endpoint function runs. From your `api.py`:

```python
class LoginRequest(BaseModel):
    customer_id: int
    password: str
```

That's it. No manual `if not isinstance(...)` checks anywhere in your code -- the type hints alone are the validation rules. This is why the malformed request above came back as a clean `422` with a descriptive error, instead of your code crashing with an unhandled exception somewhere deep inside `auth_login()`.

---
# Part 3: FastAPI -- Validation, Docs, and What Makes It Different

FastAPI's two headline features, both of which you used without extra work: automatic validation (above) and automatic interactive documentation.

## 3.1 The Auto-Generated Docs Aren't Magic, They're Derived From Your Code

`/docs` isn't a separate thing you wrote, it's generated by FastAPI reading your route definitions, Pydantic models, and type hints, and building an OpenAPI spec from them automatically. Let's fetch that raw spec directly.

In [5]:
spec = requests.get(f"{API_BASE}/openapi.json").json()

print("API title:", spec["info"]["title"])
print("\nEndpoints FastAPI discovered from your code:")
for path, methods in spec["paths"].items():
    for method in methods:
        print(f"  {method.upper():6s} {path}")

# The Swagger UI page at /docs is just a frontend that reads and renders THIS json.
# Same idea as your React app rendering data from your own API -- it's client-rendered-
# from-JSON all the way down.

API title: DRAM Express Assistant API

Endpoints FastAPI discovered from your code:
  POST   /api/login
  POST   /api/logout
  POST   /api/chat
  GET    /api/orders
  POST   /api/review


## 3.2 Async Endpoints (Why You'll See `async def` in FastAPI Code Elsewhere)

Your `api.py` uses plain `def` for its endpoints, which is completely valid, FastAPI runs them fine. You'll often see `async def` in other FastAPI code though, worth understanding why: an `async` endpoint can pause while waiting on something slow (a database call, another API request) and let the server handle *other* requests in that gap, instead of blocking entirely.

Your endpoints don't use `async` because `llama_cpp`'s inference calls are not natively async-compatible without extra wrapping, so `def` (which FastAPI runs in a thread pool automatically) was the simpler, correct choice here, not a mistake.

---
# Part 4: React Fundamentals

This part can't execute here (no JS kernel), so it walks through the real code from your project with explanation.

## 4.1 Components and Props

A React component is a function that returns UI. Props are how a parent passes data down to a child component. From your `App.jsx`:

```jsx
<ReviewForm sessionId={sessionId} />
```

`App` owns the `sessionId` state. It hands a copy down to `ReviewForm` as a prop, so `ReviewForm` can use it (to call `/api/orders?session_id=...`) without needing to know or care how that session was created. This is the core React pattern: state lives in one place, and gets handed down to whatever needs it.

## 4.2 `useState`: Component Memory

```jsx
const [messages, setMessages] = useState([...])
```

`useState` gives a component a piece of memory that survives between renders. `messages` is the current value, `setMessages` is the *only* correct way to change it. Calling `setMessages(...)` tells React "this changed, please re-render anything that depends on it" -- you never manually touch the DOM to add a new chat bubble, you just update the state and React figures out what needs to change on screen.

This is the single biggest conceptual shift from plain JS: you stop thinking "how do I update the page," and start thinking "what should the state be right now".

## 4.3 `useEffect`: Doing Something When a Component Appears (or a Value Changes)

From `ReviewForm.jsx`:

```jsx
useEffect(() => {
    fetch(`${API_BASE}/api/orders?session_id=${sessionId}`)
        .then((res) => res.json())
        .then(setOrders)
}, [sessionId])
```

This says: "run this fetch once when the component first appears, and again any time `sessionId` changes." The `[sessionId]` array is the dependency list -- React compares it between renders, and only re-runs the effect if something in that list actually changed. Without it, the fetch would either never re-run (stale data) or run on every single render (a request storm) -- the dependency array is what makes it run exactly when it should.

## 4.4 Controlled Inputs

```jsx
<input value={input} onChange={(e) => setInput(e.target.value)} />
```

In plain HTML, an input manages its own text internally. In React, this input's value is *entirely* driven by state (`input`) -- every keystroke calls `setInput`, which updates state, which re-renders the input with the new value. This is called a "controlled component." It feels redundant at first, but it's what makes it trivial to do things like clear the input after sending a message (`setInput('')`) or validate it as the user types, since the component's displayed value and your JS state are always the same thing.

---
# Part 5: The Client-Server Boundary -- What Actually Broke

This is the concept behind the real bug you hit: the chat window hanging on "Thinking..." forever.

## 5.1 Where Your React Code Actually Runs

This is the single most important mental model for full-stack development: **your React code, once built, runs entirely inside the user's browser** -- not on your server, not on the remote machine, on whatever computer opened the page. Streamlit doesn't have this distinction (the whole thing runs server-side), which is exactly why this concept never came up before this project.

That means every `fetch("http://localhost:8000/...")` call in your React code is a request from *the browser* to *port 8000* -- and if the browser can't reach that port, the request fails, no matter how correctly your backend code is written.

## 5.2 Why This Specifically Broke on a Remote Machine

On your local laptop, `localhost:8000` naturally refers to "this same computer." On a remote SSH session, that's no longer true by default, your backend runs on the remote machine, but your browser (and therefore your React code) runs locally. VS Code's port forwarding is what bridges this: it makes a request to your local `localhost:8000` transparently tunnel over SSH to the remote machine's port 8000.

**Both ports needed forwarding independently** -- 5173 to load the React page itself, and *separately* 8000 for every API call that page's JavaScript makes afterward. Forwarding one without the other looks like it's working (the page loads fine) right up until you interact with anything that talks to the backend.

In [6]:
try:
    requests.get("http://localhost:9999/api/orders", timeout=2)
except requests.exceptions.ConnectionError as e:
    print("This is the exact class of error your unpatched fetch() calls were hitting silently:")
    print(type(e).__name__)

# The original bug: your first LoginForm/ChatWindow had no try/catch around fetch(),
# so this exact kind of failure just vanished -- no error shown, request just never
# resolved the way the code expected. Wrapping it in try/catch turned an invisible
# failure into a message the user (you) could actually see and act on.

This is the exact class of error your unpatched fetch() calls were hitting silently:
ConnectionError


---
# Part 6: Sessions, State, and Security

HTTP is stateless by design -- every request is independent, the server doesn't inherently remember who you are between them. Sessions are how you layer identity on top of that.

## 6.1 How Your Session System Works

1. `/api/login` succeeds -> server generates a random `session_id` (a UUID) and stores `{sql_conn, customer_id, first_name}` in an in-memory dict, keyed by that ID
2. The `session_id` is sent back to the browser
3. Every subsequent request (chat, orders, review) includes that `session_id`
4. The server looks it up in the dict to know who's asking

This is a deliberately simple version of a pattern used everywhere in real web apps, just usually with the session data in something more durable than a plain Python dict (Redis, a database table, or a signed token like a JWT that doesn't need server-side storage at all).

In [7]:
session_a = requests.post(f"{API_BASE}/api/login", json={"customer_id": 1, "password": "123"}).json()["session_id"]
session_b = requests.post(f"{API_BASE}/api/login", json={"customer_id": 4, "password": "123"}).json()["session_id"]

print("Session A:", session_a)
print("Session B:", session_b)
print("Different session_ids:", session_a != session_b)

orders_a = requests.get(f"{API_BASE}/api/orders", params={"session_id": session_a}).json()
orders_b = requests.get(f"{API_BASE}/api/orders", params={"session_id": session_b}).json()

print("\nCustomer 1's orders:", [o['order_id'] for o in orders_a])
print("Customer 4's orders:", [o['order_id'] for o in orders_b])
print("\nNo overlap:", set(o['order_id'] for o in orders_a).isdisjoint(o['order_id'] for o in orders_b))

Session A: ecca044d-a9aa-4b9b-aaed-3023bd961be8
Session B: 02f2d6a6-b42a-45a8-9dd2-d26857ccef4d
Different session_ids: True

Customer 1's orders: [96]
Customer 4's orders: []

No overlap: True


## 6.2 The Real Limitation Worth Knowing

The in-memory dict approach has two honest limitations, worth naming even though they don't matter at this project's scale:

1. **It resets on server restart.** Every `--reload` during development wipes all active sessions, everyone gets logged out. Fine for a demo, not acceptable for production.
2. **It doesn't scale past one server process.** If you ever ran two copies of this API behind a load balancer, session B's data would only exist in whichever process handled that login, a request routed to the other process would find nothing.

The production fix for both is external session storage (Redis is the common choice) or moving to stateless tokens (JWTs) that encode the session data directly in something the client holds, so the server doesn't need to remember anything at all.

---
## Summary

| Concept | Where it showed up in your project |
|---|---|
| HTTP methods/status codes | Every endpoint in `api.py` |
| REST resource design | `/api/orders`, `/api/login`, `/api/review` |
| Automatic validation | Pydantic models (`LoginRequest`, `ChatRequest`, etc.) |
| Auto-generated docs | `/docs`, derived from the same models and route code |
| Component state | `useState` in every React component |
| Effects/data fetching | `useEffect` in `ReviewForm.jsx` |
| Controlled inputs | Every form field in `LoginForm.jsx`, `ChatWindow.jsx` |
| Client-server boundary | The actual "Thinking..." bug you hit and fixed |
| Session management | The in-memory `sessions` dict in `api.py` |

You built a real, working version of every piece here, and the bug you hit (the client-server boundary one) is genuinely one of the most common early full-stack mistakes -- understanding exactly why it happened is worth more than if everything had just worked on the first try.